# MPNranker2 — Model Architecture

Source: [`mpnranker2.py`](../mpnranker2.py)

## Architecture Overview

```
SMILES ──► DMPNN Encoder ──► enc  (encoder_size)
                                │
extra_features ─────────────────┤
sys_features  ──► [hidden_pv layers] ──► enc_pv
                                │
             cat[enc, extra, enc_pv] ──► [hidden layers] ──► ROI  (scalar)
```

**Two modes of operation:**
- **Pairwise ranking** — given two molecules (A, B), predict which elutes later
- **Triplet learning** — given (anchor, positive, negative), also apply `TripletMarginLoss`

**Loss:**  
`total_loss = MarginRankingLoss + 0.1 × TripletMarginLoss`

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
from mpnranker2 import MPNranker

## Instantiating the Model

In [ ]:
# Default architecture matching the published model
ranker = MPNranker(
    encoder='dmpnn',
    extra_features_dim=0,      # additional molecular descriptors (e.g., logP)
    sys_features_dim=12,       # chromatographic system features (HSM + Tanaka + pH)
    hidden_units=[256, 64],    # ranking head layer sizes
    hidden_units_pv=[256, 256],# system×molecule preference encoding layers
    encoder_size=512,          # DMPNN hidden size
    depth=3,                   # message-passing steps
    dropout_rate_encoder=0.0,
    dropout_rate_pv=0.0,
    dropout_rate_rank=0.0,
    res_conn_enc=False,        # residual connection from encoder to ranking head
    no_sigmoid_roi=True,       # don't clamp ROI to [0,1]
)
print(ranker)
total = sum(p.numel() for p in ranker.parameters())
trainable = sum(p.numel() for p in ranker.parameters() if p.requires_grad)
print(f'\nTotal params:     {total:,}')
print(f'Trainable params: {trainable:,}')

## Forward Pass Details

In [ ]:
import inspect
print(inspect.getsource(MPNranker.forward))

### Key points from `forward()`:

1. **Input** `batch` is a tuple of 1–3 `(graphs, extra, sysf)` tuples (for pair or triplet).
2. Each molecule is encoded by the DMPNN → `enc` vector.
3. `enc`, `extra`, and `sysf` are concatenated and passed through **preference encoding layers** (`hidden_pv`) to get `enc_pv`.
4. Final ranking layers take `[enc, extra, enc_pv]` → single **ROI** scalar.
5. Returns list of `{'embedding': ..., 'roi': ...}` dicts (one per molecule in the batch).

## Loss Function

In [ ]:
print(inspect.getsource(MPNranker.loss_step))

### Loss breakdown:

| Component | Formula | Weight |
|---|---|---|
| `MarginRankingLoss` | `max(0, -y*(roi_a - roi_p) + margin)` per pair | 1.0 |
| `TripletMarginLoss` | `max(0, d(a,p) - d(a,n) + margin)` on embeddings | 0.1 |

Triplet loss is only applied when 3 molecules are in the batch (triplet mode).

## Predict Method

In [ ]:
print(inspect.getsource(MPNranker.predict))

## Loading a Saved Model

In [ ]:
from evaluate import load_model

model = load_model('../models/2-step0525.pt', all_in_one=True)
model.eval()

# Patch dropout for inference
for enc in model.encoder.encoder:
    if not hasattr(enc, 'dropout_layer'):
        enc.dropout_layer = nn.Dropout(p=0.0)

print('Encoder size:', model.encoder.encoder[0].W_h.out_features)
print('Sys features dim:', model.sys_features_dim)
print('Extra features dim:', model.extra_features_dim)

## `custom_collate` — Batch Collation

The `DataLoader` uses a custom collate function to handle variable-length molecular graphs.
It supports both **pair** (2 molecules) and **triplet** (3 molecules) batches.

In [ ]:
from mpnranker2 import custom_collate
print(inspect.getsource(custom_collate))